# CRISP-DM Phase 4 — Modeling (Random Forest, 3 classes)

Trains and evaluates the **Random Forest** on the 3-class Apple-Watch data using the
current pipeline modules, with **leave-one-set-out** cross-validation (ADR-021) — the
honest, leakage-free estimate for a single-subject dataset. This reproduces the
methodology of `scripts/train_model.py`.

> This notebook **does not overwrite** the committed `best_model.joblib`; it fits models
> in memory only. Expect a few minutes of runtime for the 75-fold CV.

ADRs: 009 (model choice), 016 (classes), 018 (features), 019 (augmentation), 021 (eval).

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import LeaveOneGroupOut

from ml4b.data.augmentation import augment_windows
from ml4b.data.canonical import OVERLAP, TARGET_HZ, WINDOW_SIZE
from ml4b.data.features_invariant import extract_invariant_features, feature_columns
from ml4b.data.kaggle_loader import TARGET_CLASSES, load_kaggle_3class
from ml4b.data.windowing import apply_sliding_window
from ml4b.models.train import train_random_forest
from ml4b.utils.metrics import load_model_metrics

CLASS_NAMES = sorted(TARGET_CLASSES)
N_AUGMENT = 5  # 6x total, same as scripts/train_model.py (ADR-019)
print("classes:", CLASS_NAMES)

## 1. Build the feature matrix (load -> window -> augment -> features)
Augmented copies keep their source `recording_id` so they can be excluded from the
held-out fold. `is_augmented` marks them (originals come first).

In [ ]:
raw_df = load_kaggle_3class()
window_df = apply_sliding_window(
    raw_df, window_size=WINDOW_SIZE, overlap=OVERLAP, sampling_rate=TARGET_HZ
)
n_orig = len(window_df)
combined = augment_windows(window_df, n_augment=N_AUGMENT, random_state=42)
t0 = time.time()
feats = extract_invariant_features(combined)
feature_names = feature_columns(feats)
is_augmented = np.arange(len(feats)) >= n_orig
X = feats[feature_names].to_numpy()
y = feats["exercise_name"].to_numpy()
groups = feats["recording_id"].to_numpy()
print(
    f"original windows: {n_orig:,} | with augmentation: {len(feats):,} | features: {len(feature_names)}"
)
print(f"feature extraction: {time.time() - t0:.1f}s")

## 2. Leave-one-set-out cross-validation
For each held-out set: train on all other sets (incl. their augmented copies), test on
the held-out set's **original** windows only. No leakage through sets or augmentation.

In [ ]:
logo = LeaveOneGroupOut()
y_true, y_pred = [], []
t0 = time.time()
folds = logo.get_n_splits(groups=groups)
for i, (tr, te) in enumerate(logo.split(X, y, groups)):
    te = te[~is_augmented[te]]  # test on original held-out windows only
    if len(te) == 0:
        continue
    model = train_random_forest(X[tr], y[tr], random_state=42)
    y_pred.extend(model.predict(X[te]).tolist())
    y_true.extend(y[te].tolist())
    if (i + 1) % 15 == 0:
        print(f"  {i + 1}/{folds} folds done...")
print(f"leave-one-set-out CV finished in {time.time() - t0:.0f}s over {folds} folds")

In [ ]:
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=CLASS_NAMES)
acc = accuracy_score(y_true, y_pred)
per_class = f1_score(y_true, y_pred, average=None, labels=CLASS_NAMES, zero_division=0)
print(f"Macro F1 (leave-one-set-out): {macro_f1:.4f}")
print(f"Accuracy                    : {acc:.4f}")
print("Per-class F1:")
for c, f in zip(CLASS_NAMES, per_class):
    print(f"  {c:<18} {f:.4f}")

## 3. Confusion matrix (row-normalized)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES).astype(float)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
for r in range(len(CLASS_NAMES)):
    for col in range(len(CLASS_NAMES)):
        ax.text(
            col,
            r,
            f"{cm_norm[r, col]:.2f}",
            ha="center",
            va="center",
            color="white" if cm_norm[r, col] > 0.5 else "black",
        )
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 4. Per-class F1

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(CLASS_NAMES, per_class, color=["#4C78A8", "#54A24B", "#E45756"])
ax.axhline(0.80, ls="--", color="red", label="target 0.80")
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1")
ax.set_title("Per-class F1 (leave-one-set-out)")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Cross-check against the committed metrics
The shipped model (`scripts/train_model.py`) writes its honest metrics to
`models/saved/model_metrics.json`. The notebook's CV reproduces the same methodology,
so the numbers should match closely.

In [ ]:
m = load_model_metrics()
print(f"committed macro F1 : {m['cv_macro_f1']:.4f}  (notebook: {macro_f1:.4f})")
print(f"committed accuracy : {m['cv_accuracy']:.4f}  (notebook: {acc:.4f})")
print("committed per-class F1:", m["cv_per_class_f1"])

## 6. Final model
For deployment, `scripts/train_model.py` trains the Random Forest on **all** sets +
augmentation and saves it to `models/saved/best_model.joblib` (committed). This notebook
deliberately does **not** overwrite that artifact — it only validates the methodology.

## 7. Summary
- Random Forest, leave-one-set-out **macro F1 ~ 0.78** on a single-subject anchor.
- Bicep curl vs tricep extension is the most-confused pair (both elbow movements).
- Honest limits are discussed in **05_evaluation**.